# Busca documental no corpus de Sherlock Holmes — M4

Este notebook executa a avaliação final congelada do mecanismo de busca literal e TF-IDF sobre o corpus de nove obras. Clone e ambiente Python são preparados em uma pasta isolada, sem substituir as bibliotecas do kernel.

## Summary (English)

This notebook runs the frozen final evaluation of lexical and TF-IDF retrieval over the nine-work Sherlock Holmes corpus. It reports known-reference retrieval metrics, the final query-level labels and scores, corpus totals, and runtime versions, then saves `m4_final.json`. Known references are a limited benchmark, not an exhaustive accuracy measure or relevance judgment. The final set is not used for tuning.


## Entradas e configuração

- Dataset: [Sherlock Books no Kaggle](https://www.kaggle.com/datasets/talesgomes27/sherleck-books).
- No Kaggle, habilite **Internet** para clonar o projeto e instalar `uv` no kernel.
- Anexe o dataset. A busca automática exige exatamente um `a_study_in_scarlet.txt` em `/kaggle/input`; em execução local, `SHERLOCK_RAW_DIR` pode apontar para `data/raw`.
- `SHERLOCK_NOTEBOOK_WORK_DIR` define o workspace local (padrão Kaggle: `/kaggle/working/sherlock-m4`).

A avaliação final não altera core, dependências, rótulos ou o limiar TF-IDF congelado. Não publica nem envia resultados e não requer credenciais.


In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys

WORK_DIR = Path(os.environ.get("SHERLOCK_NOTEBOOK_WORK_DIR", "/kaggle/working/sherlock-m4")).expanduser().resolve()
SOURCE_DIR = WORK_DIR / "source"
REPORT_PATH = WORK_DIR / "m4_final.json"
LOCAL_RAW = os.environ.get("SHERLOCK_RAW_DIR")
if LOCAL_RAW:
    RAW_DIR = Path(LOCAL_RAW).expanduser().resolve()
else:
    candidates = list(Path("/kaggle/input").rglob("a_study_in_scarlet.txt")) if Path("/kaggle/input").exists() else []
    if len(candidates) != 1:
        raise RuntimeError(f"Esperado exatamente um a_study_in_scarlet.txt em /kaggle/input; encontrados {len(candidates)}")
    RAW_DIR = candidates[0].parent

WORK_DIR.mkdir(parents=True, exist_ok=True)
UV_ENV = os.environ.copy()
UV_ENV.pop("VIRTUAL_ENV", None)
UV_ENV.pop("UV_PROJECT_ENVIRONMENT", None)
UV_ENV["UV_CACHE_DIR"] = str(WORK_DIR / "cache")
UV_ENV["UV_PYTHON_INSTALL_DIR"] = str(WORK_DIR / "python")
Path(UV_ENV["UV_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
Path(UV_ENV["UV_PYTHON_INSTALL_DIR"]).mkdir(parents=True, exist_ok=True)

# Keep command logs useful without revealing local home/workspace paths.
def run_quiet(command, *, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.returncode:
        details = (result.stdout + "\n" + result.stderr).replace(str(WORK_DIR), "<workspace>").replace(str(RAW_DIR), "<dataset>")
        details = details.replace(str(Path.home()), "<home>")
        raise RuntimeError(f"Comando falhou ({result.returncode}): {command[0]}\n{details[-5000:]}")
    return result

print("Workspace preparado.")
print("Dataset encontrado:", bool(RAW_DIR.is_dir()))


Workspace preparado.
Dataset encontrado: True


In [2]:
UV_VERSION = "0.13.5"
probe = subprocess.run([sys.executable, "-m", "uv", "--version"], text=True, capture_output=True)
parts = probe.stdout.strip().split()
if probe.returncode or len(parts) < 2 or parts[1] != UV_VERSION:
    run_quiet([sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"])

SOURCE_URL = "https://github.com/talesgomes27/sherlock-document-search.git"
SUPPORT_COMMIT = "51a69a0264ef318d8506da0a8db627c295a5ccba"
if SOURCE_DIR.exists():
    if not (SOURCE_DIR / ".git").exists():
        raise RuntimeError("Workspace de source existe sem checkout Git; escolha outro workspace.")
    head = run_quiet(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"]).stdout.strip()
    dirty = run_quiet(["git", "-C", str(SOURCE_DIR), "status", "--porcelain", "--ignore-submodules=all"]).stdout.strip()
    if head != SUPPORT_COMMIT or dirty:
        raise RuntimeError("Checkout existente não corresponde ao commit de suporte limpo; escolha outro workspace.")
else:
    run_quiet(["git", "clone", "--no-checkout", SOURCE_URL, str(SOURCE_DIR)])
    run_quiet(["git", "-C", str(SOURCE_DIR), "checkout", "--detach", SUPPORT_COMMIT])

actual = run_quiet(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"]).stdout.strip()
assert actual == SUPPORT_COMMIT, "Commit clonado diverge do pin informado."
print("Checkout de suporte validado:", actual)


Checkout de suporte validado: 51a69a0264ef318d8506da0a8db627c295a5ccba


In [3]:
run_quiet([
    sys.executable, "-m", "uv", "sync", "--managed-python", "--python", "3.11.17", "--frozen",
], cwd=SOURCE_DIR, env=UV_ENV)
version_json = run_quiet([
    sys.executable, "-m", "uv", "run", "--managed-python", "--python", "3.11.17", "--frozen",
    "python", "-c", "import json,sys,sklearn,numpy,scipy; print(json.dumps({'python':sys.version.split()[0],'scikit-learn':sklearn.__version__,'numpy':numpy.__version__,'scipy':scipy.__version__}))",
], cwd=SOURCE_DIR, env=UV_ENV).stdout.strip()
VERSIONS = json.loads(version_json)
print("Ambiente isolado sincronizado.")


Ambiente isolado sincronizado.


## Relatório de desenvolvimento e preflight congelado

O relatório M3 é lido do caminho conhecido e apenas suas métricas agregadas são mostradas. Antes de iniciar as consultas finais, verificamos o snapshot congelado do projeto e registramos as versões do ambiente isolado.


In [4]:
from IPython.display import display, Markdown

METRIC_KEYS = (
    "positive_queries", "positive_hits", "known_hit_at_5",
    "negative_queries", "false_positive_queries", "false_positive_rate",
)

def metrics_markdown(title, section):
    rows = ["| Métrica | Resultado |", "|---|---:|"]
    for key in METRIC_KEYS:
        value = section.get(key, "—")
        if key == "false_positive_rate" and isinstance(value, (int, float)):
            value = f"{value:.4f} ({value:.1%})"
        rows.append(f"| `{key}` | {value} |")
    display(Markdown(f"### {title}\n\n" + "\n".join(rows)))

DEV_REPORT = SOURCE_DIR / "reports" / "m3_development.json"
if DEV_REPORT.is_file():
    dev_report = json.loads(DEV_REPORT.read_text(encoding="utf-8"))
    for name in ("literal", "tfidf"):
        metrics_markdown(f"Desenvolvimento M3 — {name}", dev_report[name])
else:
    print("Relatório M3 não incluído no commit de suporte.")

freeze_check = run_quiet([
    sys.executable, "-m", "uv", "run", "--managed-python", "--python", "3.11.17", "--frozen",
    "python", "-m", "sherlock_search.development", "--check-freeze",
], cwd=SOURCE_DIR, env=UV_ENV)
print("Preflight freeze: aprovado.")

display(Markdown("### Versões do runtime\n\n| Componente | Versão |\n|---|---:|\n" + "\n".join(f"| `{key}` | {value} |" for key, value in VERSIONS.items())))


### Desenvolvimento M3 — literal

| Métrica | Resultado |
|---|---:|
| `positive_queries` | 8 |
| `positive_hits` | 3 |
| `known_hit_at_5` | 0.375 |
| `negative_queries` | 2 |
| `false_positive_queries` | 0 |
| `false_positive_rate` | 0.0000 (0.0%) |

### Desenvolvimento M3 — tfidf

| Métrica | Resultado |
|---|---:|
| `positive_queries` | 8 |
| `positive_hits` | 1 |
| `known_hit_at_5` | 0.125 |
| `negative_queries` | 2 |
| `false_positive_queries` | 2 |
| `false_positive_rate` | 1.0000 (100.0%) |

Preflight freeze: aprovado.


### Versões do runtime

| Componente | Versão |
|---|---:|
| `python` | 3.11.17 |
| `scikit-learn` | 1.9.1 |
| `numpy` | 2.4.6 |
| `scipy` | 1.17.1 |

## Avaliação final congelada

A avaliação calcula os resultados uma única vez, sem ajustes ou calibração. As consultas são divididas pelos rótulos answerable/não answerable; os acertos são relativos apenas às referências conhecidas.


A preparação valida os hashes dos nove TXT, trata BOM e quebras de linha em memória e usa os recortes editoriais revisados no M1. Cada parágrafo conserva arquivo e linhas de origem. O índice usa todo o acervo como catálogo de busca; os rótulos finais não orientam a preparação nem os parâmetros. Consulte os créditos e as pendências de uso em `docs/provenance.md` do repositório fixado.

In [5]:
run_quiet([
    sys.executable, "-m", "uv", "run", "--managed-python", "--python", "3.11.17", "--frozen",
    "python", "-m", "sherlock_search.final_evaluation",
    "--root", str(SOURCE_DIR), "--raw-dir", str(RAW_DIR), "--output", str(REPORT_PATH),
], cwd=SOURCE_DIR, env=UV_ENV)
assert REPORT_PATH.is_file(), "O módulo não criou o arquivo de avaliação final."
final_report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
results = final_report["results"]
assert final_report["split"] == "final" and final_report["query_count"] == 20
assert len(results) == 20, f"Esperadas 20 consultas finais; encontradas {len(results)}"
assert sum(bool(item["answerable"]) for item in results) == final_report["literal"]["positive_queries"]
assert sum(not bool(item["answerable"]) for item in results) == final_report["literal"]["negative_queries"]
corpus_totals = final_report["corpus"]["totals"]
file_count = corpus_totals["files"]
assert file_count == 9, f"Esperadas 9 obras no corpus; total informado: {file_count}"
print("JSON final criado; 20 consultas e 9 obras validadas.")


quality_labels = {
    "files": "Arquivos", "size_bytes": "Bytes nos TXT", "source_lines": "Linhas de origem",
    "source_words": "Palavras nos originais", "retained_words": "Palavras retidas",
    "removed_words": "Palavras excluídas pelos recortes", "paragraphs": "Parágrafos",
}
quality_rows = ["| Medida | Quantidade |", "|---|---:|"]
quality_rows += [f"| {label} | {corpus_totals[key]:,} |" for key, label in quality_labels.items()]
display(Markdown("### Corpus e preparação — M1\n\n" + "\n".join(quality_rows)))


JSON final criado; 20 consultas e 9 obras validadas.


### Corpus e preparação — M1

| Medida | Quantidade |
|---|---:|
| Arquivos | 9 |
| Bytes nos TXT | 4,041,127 |
| Linhas de origem | 80,571 |
| Palavras nos originais | 686,450 |
| Palavras retidas | 656,884 |
| Palavras excluídas pelos recortes | 29,566 |
| Parágrafos | 15,598 |

In [6]:
for name in ("literal", "tfidf"):
    metrics_markdown(f"Avaliação final — {name}", final_report[name])

rows = ["| ID | Rótulo | Hit conhecido — literal | Ocorrências top 1 | N literal | Hit conhecido — TF-IDF | Cosseno top 1 | N TF-IDF |", "|---|---|---:|---:|---:|---:|---:|---:|"]
for item in results:
    known = set(item["known_relevant_ids"])
    literal = item["literal"]
    tfidf = item["tfidf"]
    lit_hit = ("sim" if any(entry["id"] in known for entry in literal[:5]) else "não") if item["answerable"] else "—"
    tfidf_hit = ("sim" if any(entry["id"] in known for entry in tfidf[:5]) else "não") if item["answerable"] else "—"
    lit_score = literal[0]["score"] if literal else "—"
    tfidf_score = format(tfidf[0]["score"], ".4f") if tfidf else "—"
    rows.append(f"| {item['id']} | {'com resposta' if item['answerable'] else 'sem resposta'} | {lit_hit} | {lit_score} | {len(literal)} | {tfidf_hit} | {tfidf_score} | {len(tfidf)} |")
display(Markdown("\n".join(rows)))

print("Nas consultas sem resposta, qualquer resultado conta como retorno indevido; o hit conhecido não se aplica.")

### Avaliação final — literal

| Métrica | Resultado |
|---|---:|
| `positive_queries` | 16 |
| `positive_hits` | 7 |
| `known_hit_at_5` | 0.4375 |
| `negative_queries` | 4 |
| `false_positive_queries` | 0 |
| `false_positive_rate` | 0.0000 (0.0%) |

### Avaliação final — tfidf

| Métrica | Resultado |
|---|---:|
| `positive_queries` | 16 |
| `positive_hits` | 8 |
| `known_hit_at_5` | 0.5 |
| `negative_queries` | 4 |
| `false_positive_queries` | 4 |
| `false_positive_rate` | 1.0000 (100.0%) |

| ID | Rótulo | Hit conhecido — literal | Ocorrências top 1 | N literal | Hit conhecido — TF-IDF | Cosseno top 1 | N TF-IDF |
|---|---|---:|---:|---:|---:|---:|---:|
| final-01 | com resposta | sim | 1 | 2 | sim | 0.3422 | 2 |
| final-02 | com resposta | sim | 3 | 5 | sim | 0.5407 | 5 |
| final-03 | com resposta | não | — | 0 | não | 0.7486 | 5 |
| final-04 | com resposta | sim | 1 | 1 | sim | 0.6921 | 5 |
| final-05 | com resposta | não | — | 0 | não | 0.4529 | 5 |
| final-06 | com resposta | não | — | 0 | não | 0.3519 | 5 |
| final-07 | com resposta | não | — | 0 | não | 0.3979 | 5 |
| final-08 | com resposta | não | — | 0 | sim | 0.4185 | 5 |
| final-09 | com resposta | não | — | 0 | não | 0.2802 | 5 |
| final-10 | com resposta | não | — | 0 | não | 0.5135 | 5 |
| final-11 | com resposta | não | — | 0 | não | 0.4139 | 5 |
| final-12 | com resposta | sim | 1 | 1 | sim | 0.4195 | 5 |
| final-13 | com resposta | não | 1 | 5 | sim | 0.5906 | 5 |
| final-14 | com resposta | sim | 1 | 1 | sim | 0.3824 | 5 |
| final-15 | com resposta | sim | 3 | 5 | não | 0.8776 | 5 |
| final-16 | com resposta | sim | 1 | 1 | sim | 0.6942 | 5 |
| final-17 | sem resposta | — | — | 0 | — | 0.6442 | 5 |
| final-18 | sem resposta | — | — | 0 | — | 0.3807 | 5 |
| final-19 | sem resposta | — | — | 0 | — | 0.3781 | 5 |
| final-20 | sem resposta | — | — | 0 | — | 0.5080 | 5 |

Nas consultas sem resposta, qualquer resultado conta como retorno indevido; o hit conhecido não se aplica.


In [7]:
postflight = run_quiet([
    sys.executable, "-m", "uv", "run", "--managed-python", "--python", "3.11.17", "--frozen",
    "python", "-m", "sherlock_search.development", "--check-freeze",
], cwd=SOURCE_DIR, env=UV_ENV)
print("Postflight freeze: aprovado.")
print("Arquivo JSON exportado para o workspace.")


Postflight freeze: aprovado.
Arquivo JSON exportado para o workspace.


## Limitações e interpretação

- Referências conhecidas são limitadas e não medem accuracy nem representam julgamentos exaustivos de relevância.
- A taxa de falso positivo depende das consultas negativas fornecidas; algumas podem ser anacrônicas.
- Esta execução final não altera parâmetros nem usa os resultados para tuning.
- As tabelas não exibem texto das obras; apenas IDs, rótulos, scores e métricas.
- `m4_final.json` fica no workspace para inspeção/download manual. Nenhum upload ou publicação é iniciado.

- Desenvolvimento e final usam consultas diferentes; a diferença entre suas taxas não representa aprendizado ou ajuste posterior do modelo.
- Os resultados podem ser reproduzidos sem alterar os métodos; repetições para validação não autorizam selecionar novos parâmetros com base no conjunto final.